In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
import src.functions as fn
from src.functions import (
    keep_from_peak,
    length_fix,
    split_and_add,
    group_by_activity,
    group_by_prefix,
)

In [0]:
# Paths
VOLUME_PATH = CFG["data"]["volume_path"]

print("Config loaded ✅")
print(f"Volume path : {VOLUME_PATH}")
print(f"Sample rate : {CFG['data']['sample_rate']} Hz")
print(f"Target length : {CFG['data']['signal_length']} samples")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Load all arrays from volume
X_all              = np.load(f"{VOLUME_PATH}/X_all.npy",              allow_pickle=True)
y_all              = np.load(f"{VOLUME_PATH}/y_all.npy",              allow_pickle=True)
activity_codes_all = np.load(f"{VOLUME_PATH}/activity_codes_all.npy", allow_pickle=True)
file_names_all     = np.load(f"{VOLUME_PATH}/file_names_all.npy",     allow_pickle=True)

print(f"Total signals    : {X_all.shape[0]}")
print(f"Alignment check  : ", end="")

# Verify all 4505 signals are aligned
mismatches = sum(
    1 for i in range(len(file_names_all))
    if file_names_all[i].split("_")[0] != activity_codes_all[i]
)
print(f"✅ Perfect" if mismatches == 0 else f"❌ {mismatches} mismatches")

# Split into train and test based on config
TRAIN_SUBJECTS = CFG["data"]["train_subjects"]
TEST_SUBJECTS  = CFG["data"]["test_subjects"]

train_idx = [i for i, f in enumerate(file_names_all)
             if any(sub in f for sub in TRAIN_SUBJECTS)]
test_idx  = [i for i, f in enumerate(file_names_all)
             if any(sub in f for sub in TEST_SUBJECTS)]

X_train              = X_all[train_idx]
y_train              = y_all[train_idx]
activity_codes_train = activity_codes_all[train_idx]
file_names_train     = file_names_all[train_idx]

X_test               = X_all[test_idx]
y_test               = y_all[test_idx]
activity_codes_test  = activity_codes_all[test_idx]
file_names_test      = file_names_all[test_idx]

print(f"Train : {X_train.shape[0]} signals  "
      f"ADL={np.sum(y_train=='ADL')}  Fall={np.sum(y_train=='Fall')}")
print(f"Test  : {X_test.shape[0]} signals   "
      f"ADL={np.sum(y_test=='ADL')}  Fall={np.sum(y_test=='Fall')}")

In [0]:
# Split X_train into activity groups
# Falls — all F codes together
fall_data = group_by_prefix(X_train, activity_codes_train, "F")

# ADL groups
d01_data = group_by_activity(X_train, activity_codes_train, "D01")
d02_data = group_by_activity(X_train, activity_codes_train, "D02")
d03_data = group_by_activity(X_train, activity_codes_train, "D03")
d04_data = group_by_activity(X_train, activity_codes_train, "D04")
d05_data = group_by_activity(X_train, activity_codes_train, "D05")
d06_data = group_by_activity(X_train, activity_codes_train, "D06")
d07_data = group_by_activity(X_train, activity_codes_train, "D07")
d08_data = group_by_activity(X_train, activity_codes_train, "D08")
d09_data = group_by_activity(X_train, activity_codes_train, "D09")
d10_data = group_by_activity(X_train, activity_codes_train, "D10")
d11_data = group_by_activity(X_train, activity_codes_train, "D11")
d12_data = group_by_activity(X_train, activity_codes_train, "D12")
d13_data = group_by_activity(X_train, activity_codes_train, "D13")
d14_data = group_by_activity(X_train, activity_codes_train, "D14")
d15_data = group_by_activity(X_train, activity_codes_train, "D15")
d16_data = group_by_activity(X_train, activity_codes_train, "D16")
d17_data = group_by_activity(X_train, activity_codes_train, "D17")
d18_data = group_by_activity(X_train, activity_codes_train, "D18")
d19_data = group_by_activity(X_train, activity_codes_train, "D19")

# Quick summary
print(f"\nGroup sizes:")
print(f"  Falls        : {len(fall_data)}")
print(f"  D01 (walk slow)  : {len(d01_data)}")
print(f"  D02 (walk fast)  : {len(d02_data)}")
print(f"  D03 (jog slow)   : {len(d03_data)}")
print(f"  D04 (jog fast)   : {len(d04_data)}")
print(f"  D05 (stairs slow): {len(d05_data)}")
print(f"  D06 (stairs fast): {len(d06_data)}")
print(f"  D07–D19          : {sum([len(d07_data), len(d08_data), len(d09_data), len(d10_data), len(d11_data), len(d12_data), len(d13_data), len(d14_data), len(d15_data), len(d16_data), len(d17_data), len(d18_data), len(d19_data)])}")

In [0]:
print("Processing Falls...")

# Step 1 — find peak and keep 400 samples each side
fall_peaked = keep_from_peak(fall_data, window_size=400)

# Step 2 — guarantee exactly 800 samples for every signal
fall_processed = length_fix(fall_peaked, length=800)

# Step 3 — verify
shapes = set(s.shape for s in fall_processed)
print(f"\nFall signals processed : {len(fall_processed)}")
print(f"Unique shapes          : {shapes}")
print(f"Expected               : {{(6, 800)}}")
print(f"All correct            : {shapes == {(6, 800)}}")